# Export and evaluate the experimental ONNX profiles

Experimental first-pass runner. See [the bundled guide](../README.md).


Use a BF16-capable GPU with room for the **8B** teacher (A100 is the intended first debugging target). The runner does not silently quantize or replace the teacher. A T4 is not the default BF16 path. CPU offload is configurable but needs host RAM and will be slow.

Every command runs in its own Python process so the teacher is released before student training. Keep the same config and Drive run directory across notebooks. If pip upgrades packages already imported by Colab, restart the runtime and rerun setup.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
# Upload the elastic_embed_colab folder to My Drive. Change this if nested elsewhere.
BUNDLE = Path('/content/drive/MyDrive/elastic_embed_colab')
assert (BUNDLE / 'pyproject.toml').is_file(), f'Upload the complete bundle to {BUNDLE}'
os.chdir(BUNDLE)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Keep model downloads on local Colab disk; caches/checkpoints go to Drive.
os.environ.setdefault('HF_HOME', '/content/hf-cache')
subprocess.run(['bash', str(BUNDLE / 'scripts/bootstrap_colab.sh')],
               env={**os.environ, 'COLAB_PYTHON': sys.executable}, check=True)


In [ ]:
import yaml
CONFIG = BUNDLE / 'runs/smoke-mohawk-config.yaml'
CONFIG.parent.mkdir(parents=True, exist_ok=True)
if not CONFIG.exists():
    cfg = yaml.safe_load((BUNDLE / 'configs/smoke.yaml').read_text())
    # Set these BEFORE prepare/cache/train; resume checks enforce matching identities.
    cfg['run_dir'] = str(BUNDLE / 'runs/smoke-mohawk')
    cfg['data']['sources'] = str(BUNDLE / 'configs/sources.yaml')
    # Default smoke budget is 20k. For a smaller wiring check, explicitly lower both:
    # cfg['data']['max_examples'] = 1800
    # cfg['data']['min_per_language'] = 20
    # cfg['training'].update(full_depth_steps=5, prefix_steps=5, matryoshka_steps=5, effective_batch=8)
    # cfg['mohawk'].update(examples=32, validation_examples=18, matrix_steps=3, block_steps=5, effective_batch=8, block_freeze_ffn_steps=2)
    CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(CONFIG.read_text())

def run(command, *args):
    import sys, time
    log_dir = CONFIG.parent / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f'{command}-{time.time_ns()}.log'
    argv = [sys.executable, '-u', '-m', 'elastic_embed.cli', command, '--config', str(CONFIG), *args]
    print(f'Python: {sys.executable}\nLog: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{command} failed (exit {returncode}). See output above or {log_path}')


The export is an FP32 standard-operator Loop fallback with one graph per depth. It is for correctness and measurement; weights are duplicated between profiles. CPU parity does not imply browser support or useful latency.

In [ ]:
run('evaluate')
run('export')
run('benchmark')


In [ ]:
from elastic_embed.deployment import Encoder
cfg = yaml.safe_load(CONFIG.read_text())
encoder = Encoder(Path(cfg['run_dir']) / 'onnx')
vectors = encoder.encode(['A short post.', 'Una publicación breve.'], depth=3, truncate_dim=128)
print(vectors.shape, encoder.representation_id(depth=3, truncate_dim=128))
